## Imports

In [ ]:
%pip -q install -U langchain langchain-text-splitters langchain-community langchain-experimental bs4
%pip -q install -U langchain-core
%pip -q install -U "huggingface-hub>=0.33.4,<1.0.0"
%pip -q install -U "sentence-transformers>=2.2.0,<3.0.0"
%pip -q install -U langchain-huggingface
%pip -q install -U datasets ragas

import getpass
import os
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown
from langchain.agents import create_agent
from langchain.agents.middleware import ModelRequest, dynamic_prompt
from langchain_community.document_loaders import CSVLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_huggingface import HuggingFaceEmbeddings
from ragas import evaluate
import re
import ast
import langchain
import time

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


## Hugging Face (Inference API)

In [2]:
os.environ["HUGGINGFACEHUB_API_TOKEN"] = ""
if not os.environ.get("HUGGINGFACEHUB_API_TOKEN"):
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = getpass.getpass(
        "Enter Hugging Face token (HUGGINGFACEHUB_API_TOKEN): "
    )

# Allowed model list
HF_MODELS = [
    "meta-llama/Llama-3.1-8B-Instruct",
    "Qwen/Qwen2.5-72B-Instruct",
    "mistralai/Mistral-Nemo-Instruct-2407",
    "deepseek-ai/DeepSeek-R1",
]

HF_MODEL_ID = os.environ.get("HF_MODEL_ID", HF_MODELS[0])
if HF_MODEL_ID not in HF_MODELS:
    raise ValueError(f"HF_MODEL_ID must be one of {HF_MODELS}. Got: {HF_MODEL_ID!r}")

HF_PROVIDER = os.environ.get("HF_PROVIDER", "auto")
HF_MAX_NEW_TOKENS = int(os.environ.get("HF_MAX_NEW_TOKENS", "4096"))
HF_TEMPERATURE = float(os.environ.get("HF_TEMPERATURE", "0.5"))
HF_TOP_P = float(os.environ.get("HF_TOP_P", "0.95"))

# Endpoint for agent: text-generation + ChatHuggingFace wrapper
llm_agent = HuggingFaceEndpoint(
    repo_id=HF_MODEL_ID,
    task="text-generation",
    provider=HF_PROVIDER,
    max_new_tokens=HF_MAX_NEW_TOKENS,
    temperature=HF_TEMPERATURE,
    top_p=HF_TOP_P,
    return_full_text=False,
)
model = ChatHuggingFace(llm=llm_agent)

# Endpoint for RAGAS evaluation: conversational task (some providers like Novita require this)
llm = HuggingFaceEndpoint(
    repo_id=HF_MODEL_ID,
    task="conversational",
    provider=HF_PROVIDER,
    max_new_tokens=HF_MAX_NEW_TOKENS,
    temperature=HF_TEMPERATURE,
    top_p=HF_TOP_P,
    return_full_text=False,
)

print(f"Using Hugging Face model: {HF_MODEL_ID}")
print(f"HF_PROVIDER={HF_PROVIDER}")
print(f"HF_MAX_NEW_TOKENS={HF_MAX_NEW_TOKENS}")

Using Hugging Face model: meta-llama/Llama-3.1-8B-Instruct
HF_PROVIDER=auto
HF_MAX_NEW_TOKENS=4096


## Embeddings

In [3]:
# Embeddings (local)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = InMemoryVectorStore(embeddings)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

#### Data and Chunks

In [4]:
csv_path = Path("fashion.csv")

loader = CSVLoader(file_path=str(csv_path), encoding="utf-8")
docs = loader.load()

all_splits = docs

# Index chunks
_ = vector_store.add_documents(documents=all_splits)

## RAG Agent

In [5]:
# --- RAG chain via dynamic prompt (middleware) ---
# This runs retrieval automatically on every user message and injects the
# retrieved content into the model prompt.
@dynamic_prompt
def prompt_with_context(request: ModelRequest) -> str:
    last_msg = request.state["messages"][-1]

    # Be robust across different message object shapes
    last_query = getattr(last_msg, "text", None) or getattr(last_msg, "content", "")

    retrieved_docs = vector_store.similarity_search(last_query, k=3)
    docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)

    system_message = (
        "You are a helpful shopping assistant for a fashion catalog. Use the following product catalog context to answer. "
        "If the answer isn't in the context, say you don't know.\n\n"
        "Output format (IMPORTANT):\n"
        "- Return your final answer as GitHub-flavored Markdown.\n"
        "- When you list products, for each product include: ProductTitle, ProductId, Size (if present), and ImageURL.\n"
        "- Also embed the image preview using Markdown image syntax on its own line: ![ProductTitle](ImageURL)\n\n"
        "CATALOG CONTEXT:\n"
        f"{docs_content}"
    )

    return system_message

agent = create_agent(model, tools=[], middleware=[prompt_with_context])

#### Query

In [ ]:
query = "hi, do you have girls pink top?"

resp = agent.invoke({"messages": [{"role": "user", "content": query}]})
md = resp["messages"][-1].content

display(Markdown(md))

## Evaluation

### Generation Evaluation

#### Create Test Examples

In [ ]:
# Hard-coded examples for evaluation
examples = [
    {
        "query": "Do you have any pink tops for girls?",
        "answer": "Yes, we have pink tops available in the fashion catalog"
    },
    {
        "query": "What sizes are available for girls clothing?",
        "answer": "Various sizes are available depending on the product"
    },
    {
        "query": "Do you have girls shoes?",
        "answer": "Yes, we have Disney Kids Princess Heart Pink Casual Shoes in the fashion catalog"
    }
]

#### Generate Additional Examples using LLM

In [ ]:
# Create example generation prompt
example_gen_prompt = PromptTemplate.from_template(
    """Given the following document, generate a question and answer pair that could be asked about it.

Document:
{doc}

Please respond in JSON format with "query" and "answer" keys."""
)

# Generate new examples from first few documents WITH RATE LIMITING
new_examples = []
for i, doc in enumerate(docs[:3]):
    try:
        if i > 0:
            time.sleep(3)  # Wait 3 seconds between requests
        chain = example_gen_prompt | model | JsonOutputParser()
        result = chain.invoke({"doc": doc.page_content})
        new_examples.append(result)
        print(f"Generated example {i+1}")
    except Exception as e:
        print(f"Skipping example generation due to: {e}")

# Combine with hard-coded examples
examples += new_examples

print(f"Total examples: {len(examples)}")
if new_examples:
    print(f"First generated example: {new_examples[0]}")

In [ ]:
# Generate queries and ground truths for RAGAS evaluation
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
import json

# Create RAGAS-specific example generation prompt
ragas_gen_prompt = PromptTemplate.from_template(
    """You are generating evaluation examples for a RAG (Retrieval Augmented Generation) system
that answers questions about a fashion product catalog.

Based on the following document snippet from the catalog, generate a realistic user query and
its corresponding ground truth answer.

Document:
{doc}

Requirements:
1. Generate a natural, conversational query that a real customer might ask
2. The ground truth answer should be accurate and based ONLY on the information in the document
3. The answer should include specific product details like ProductTitle, ProductId, Size (if available), and ImageURL
4. Format as JSON with keys: "query" (string) and "ground_truth" (string)

Respond ONLY with valid JSON, no additional text."""
)

# Generate RAGAS evaluation examples from documents WITH RATE LIMITING
ragas_examples = []
sample_docs = docs[:10]  # Use first 10 documents to generate diverse examples

print("Generating RAGAS evaluation examples...")
for i, doc in enumerate(sample_docs):
    try:
        if i > 0:
            time.sleep(3)  # Wait 3 seconds between requests to avoid rate limiting

        chain = ragas_gen_prompt | model | JsonOutputParser()
        result = chain.invoke({"doc": doc.page_content})

        # Add retrieved contexts (the document itself) for RAGAS evaluation
        ragas_example = {
            "query": result["query"],
            "ground_truth": result["ground_truth"],
            "contexts": [doc.page_content],  # For RAGAS, we include the context
            "doc_metadata": doc.metadata if hasattr(doc, 'metadata') else {}
        }
        ragas_examples.append(ragas_example)
        print(f"Generated RAGAS example {i+1}/{len(sample_docs)}: {result['query'][:50]}...")
    except Exception as e:
        print(f"Error generating example {i+1}: {e}")
        continue

print(f"\nTotal RAGAS examples generated: {len(ragas_examples)}")
if ragas_examples:
    print(f"\nExample RAGAS evaluation entry:")
    print(f"Query: {ragas_examples[0]['query']}")
    print(f"Ground Truth: {ragas_examples[0]['ground_truth'][:100]}...")
    print(f"Contexts: {len(ragas_examples[0]['contexts'])} document(s)")


#### Run Agent on Examples

In [ ]:
# Test agent on first example
test_response = agent.invoke({"messages": [{"role": "user", "content": examples[0]["query"]}]})
print(f"Query: {examples[0]['query']}")
print(f"Agent Response: {test_response['messages'][-1].content[:200]}...")

#### Manual Evaluation with Debug Mode

In [ ]:
# Enable debug mode to see internal steps
langchain.debug = True

# Run a query in debug mode
test_response = agent.invoke({"messages": [{"role": "user", "content": examples[0]["query"]}]})

# Disable debug mode
langchain.debug = False

#### LLM-Assisted Evaluation using LLM

In [ ]:
# Get predictions from agent on all examples WITH RATE LIMITING
predictions = []
for i, example in enumerate(examples):
    if i > 0:
        time.sleep(3)  # Wait 3 seconds between requests
    try:
        resp = agent.invoke({"messages": [{"role": "user", "content": example["query"]}]})
        predictions.append({
            "query": example["query"],
            "answer": example["answer"],
            "result": resp["messages"][-1].content
        })
        print(f"Processed prediction {i+1}/{len(examples)}")
    except Exception as e:
        print(f"Error on example {i+1}: {e}")
        predictions.append({
            "query": example["query"],
            "answer": example["answer"],
            "result": "Error: Rate limited or API error"
        })

print(f"Generated {len(predictions)} predictions")

In [ ]:
import time
from langchain_core.prompts import PromptTemplate

# Create evaluation prompt
eval_prompt = PromptTemplate.from_template(
    """You are an expert evaluator. Compare the predicted answer to the expected answer.

Question: {query}
Expected Answer: {answer}
Predicted Answer: {result}

Evaluate whether the predicted answer correctly addresses the question.
Respond with either "CORRECT" or "INCORRECT" followed by a brief explanation."""
)

# Evaluate predictions against ground truth WITH RATE LIMITING
graded_outputs = []
for i, pred in enumerate(predictions):
    if i > 0:
        time.sleep(3)  # Wait 3 seconds between requests
    try:
        chain = eval_prompt | model
        grade = chain.invoke({
            "query": pred["query"],
            "answer": pred["answer"],
            "result": pred["result"]
        })
        graded_outputs.append({"text": grade.content})
        print(f"Evaluated {i+1}/{len(predictions)}")
    except Exception as e:
        graded_outputs.append({"text": f"Error evaluating: {str(e)}"})
        print(f"Error evaluating {i+1}: {e}")

print(f"Evaluation complete. Graded {len(graded_outputs)} examples")

### Retrieval Evaluation

#### Load Evaluation Ground Truth

Load evaluation queries with ground truth ProductIds for standardized retrieval evaluation.

In [6]:
import json
import numpy as np
from typing import Callable

# Evaluation settings
RETRIEVAL_K = int(os.environ.get("RETRIEVAL_K", "5"))

def load_evaluation_queries(csv_path: str = "retrieval_eval_ground_truth.csv", use_filters: bool = True):
    """
    Load evaluation queries from CSV file.

    Expected CSV columns:
    - query: The user query
    - ground_truth_product_ids: Comma-separated ProductIds (e.g., "42419,34009")
    - difficulty: easy|medium|hard
    - query_type: specific|broad|recommendation|availability
    - expected_filters: JSON string (e.g., '{"Gender":"Girls","Colour":"Pink"}')
    - notes: Explanation

    Args:
        csv_path: Path to the evaluation CSV
        use_filters: If True, use expected_filters from CSV. If False, no filters.
    """
    csv_file = Path(csv_path)
    if not csv_file.exists():
        print(f"⚠️ {csv_path} not found. Using default test queries.")
        return [
            {
                "query": "Do you have pink tops for girls?",
                "ground_truth_product_ids": ["23623", "31120"],
                "difficulty": "easy",
                "query_type": "specific",
                "filters": {"Gender": "Girls", "Colour": "Pink"} if use_filters else {},
                "notes": "Default test query"
            },
        ]

    df_eval = pd.read_csv(csv_file)
    eval_queries = []

    for _, row in df_eval.iterrows():
        # Parse ground_truth_product_ids (comma-separated string)
        gt_ids_str = str(row.get("ground_truth_product_ids", ""))
        if gt_ids_str and gt_ids_str.lower() != "nan":
            product_ids = [pid.strip() for pid in gt_ids_str.split(",") if pid.strip()]
        else:
            product_ids = []  # Negative case - no matching products

        # Parse expected_filters (JSON string)
        # Only use fields that support exact matching
        ALLOWED_FILTER_FIELDS = {"Gender", "Colour", "Category", "ProductType", "SubCategory", "Usage"}

        filters = {}
        if use_filters:
            filters_str = row.get("expected_filters", "{}")
            if pd.notna(filters_str) and filters_str:
                try:
                    filters_str = str(filters_str).replace('""', '"')
                    all_filters = json.loads(filters_str)
                except json.JSONDecodeError:
                    try:
                        all_filters = ast.literal_eval(filters_str)
                    except:
                        all_filters = {}

                # Only keep allowed filter fields (exact match fields)
                filters = {k: v for k, v in all_filters.items() if k in ALLOWED_FILTER_FIELDS}

        eval_queries.append({
            "query": row["query"],
            "ground_truth_product_ids": product_ids,
            "difficulty": row.get("difficulty", "medium"),
            "query_type": row.get("query_type", "specific"),
            "filters": filters,
            "notes": row.get("notes", ""),
        })

    filter_status = "WITH filters" if use_filters else "WITHOUT filters"
    print(f"✅ Loaded {len(eval_queries)} evaluation queries {filter_status}")

    # Show difficulty distribution
    difficulties = [q.get("difficulty", "medium") for q in eval_queries]
    print(f"   Difficulty: {difficulties.count('easy')} easy, {difficulties.count('medium')} medium, {difficulties.count('hard')} hard")

    # Count negative cases
    negative_cases = sum(1 for q in eval_queries if not q["ground_truth_product_ids"])
    if negative_cases:
        print(f"   Negative cases (no match expected): {negative_cases}")

    return eval_queries


# Load evaluation queries
eval_queries = load_evaluation_queries(use_filters=False)  # No filters for baseline (pure semantic search)

if eval_queries:
    print(f"\n📋 Sample query:")
    print(f"   Query: {eval_queries[0]['query']}")
    print(f"   Expected ProductIds: {eval_queries[0]['ground_truth_product_ids']}")
    print(f"   Difficulty: {eval_queries[0].get('difficulty', 'N/A')}")


✅ Loaded 40 evaluation queries WITHOUT filters
   Difficulty: 24 easy, 12 medium, 4 hard
   Negative cases (no match expected): 4

📋 Sample query:
   Query: Show me women's black sports shoes
   Expected ProductIds: ['34651']
   Difficulty: easy


In [7]:
### Retrieval Metrics Calculation Functions

def calculate_retrieval_metrics(
    retrieved_docs: list,
    ground_truth_ids: list[str],
    k: int = 10
) -> dict:
    """
    Calculate standard retrieval metrics.

    Args:
        retrieved_docs: List of retrieved LangChain documents
        ground_truth_ids: List of expected ProductIds (as strings)
        k: Number of top results to consider

    Returns:
        Dictionary of metrics
    """
    # Extract ProductIds from retrieved documents
    retrieved_ids = []
    for doc in retrieved_docs[:k]:
        # Try to get ProductId from document metadata or parse from content
        product_id = None

        # Try metadata first
        if hasattr(doc, 'metadata') and doc.metadata:
            product_id = str(doc.metadata.get("ProductId", ""))

        # If not in metadata, try to parse from page_content
        if not product_id and hasattr(doc, 'page_content'):
            content = doc.page_content
            if "ProductId:" in content:
                try:
                    product_id = content.split("ProductId:")[1].split("\n")[0].strip()
                except:
                    pass

        if product_id:
            retrieved_ids.append(product_id)

    # Convert ground truth to set for fast lookup
    gt_set = set(str(pid) for pid in ground_truth_ids)

    # Hit@k: Did we retrieve at least one relevant document?
    hits = [1 if rid in gt_set else 0 for rid in retrieved_ids]
    hit_at_k = 1.0 if any(hits) else 0.0

    # Recall@k: Proportion of ground truth docs retrieved
    retrieved_relevant = len(set(retrieved_ids) & gt_set)
    recall_at_k = retrieved_relevant / len(gt_set) if gt_set else 0.0

    # Precision@k: Proportion of retrieved docs that are relevant
    precision_at_k = retrieved_relevant / len(retrieved_ids) if retrieved_ids else 0.0

    # MRR: Reciprocal Rank of first relevant document
    mrr = 0.0
    for i, rid in enumerate(retrieved_ids, 1):
        if rid in gt_set:
            mrr = 1.0 / i
            break

    # MAP: Mean Average Precision
    relevant_count = 0
    precision_sum = 0.0
    for i, rid in enumerate(retrieved_ids, 1):
        if rid in gt_set:
            relevant_count += 1
            precision_sum += relevant_count / i
    map_score = precision_sum / len(gt_set) if gt_set else 0.0

    # NDCG@k: Normalized Discounted Cumulative Gain
    dcg = 0.0
    for i, rid in enumerate(retrieved_ids, 1):
        if rid in gt_set:
            dcg += 1.0 / np.log2(i + 1)

    # Ideal DCG (all relevant docs at top)
    idcg = sum(1.0 / np.log2(i + 1) for i in range(1, min(len(gt_set), k) + 1))
    ndcg_at_k = dcg / idcg if idcg > 0 else 0.0

    return {
        "hit@k": hit_at_k,
        "recall@k": recall_at_k,
        "precision@k": precision_at_k,
        "mrr": mrr,
        "map": map_score,
        "ndcg@k": ndcg_at_k,
        "retrieved_ids": retrieved_ids,
        "ground_truth_ids": list(gt_set),
        "num_retrieved": len(retrieved_ids),
        "num_relevant_retrieved": retrieved_relevant,
    }


def baseline_semantic_retrieval(query: str, k: int = 5, filters: dict = None) -> list:
    """
    Baseline retrieval using InMemoryVectorStore similarity search.
    This is what the benchmark notebook uses by default.

    Note: filters parameter is accepted for API compatibility but not used
    (baseline doesn't support filtering).
    """
    return vector_store.similarity_search(query, k=k)


def evaluate_retrieval_strategy(
    retrieval_fn: Callable,
    eval_queries: list[dict],
    k: int = 5,
    strategy_name: str = "Strategy"
) -> pd.DataFrame:
    """
    Evaluate a retrieval strategy on all evaluation queries.

    Args:
        retrieval_fn: Function that takes (query, k, filters) and returns list of docs
        eval_queries: List of evaluation query dictionaries
        k: Number of results to retrieve
        strategy_name: Name of the strategy for display

    Returns:
        DataFrame with per-query and aggregate metrics
    """
    results = []

    print(f"\n{'='*60}")
    print(f"Evaluating: {strategy_name} (k={k})")
    print(f"{'='*60}")

    for i, eq in enumerate(eval_queries, 1):
        query = eq["query"]
        gt_ids = eq["ground_truth_product_ids"]
        filters = eq.get("filters", {})
        difficulty = eq.get("difficulty", "medium")
        query_type = eq.get("query_type", "specific")

        # Run retrieval
        try:
            docs = retrieval_fn(query, k=k, filters=filters if filters else None)
            metrics = calculate_retrieval_metrics(docs, gt_ids, k=k)
            metrics["query"] = query[:50] + "..." if len(query) > 50 else query
            metrics["strategy"] = strategy_name
            metrics["difficulty"] = difficulty
            metrics["query_type"] = query_type
            results.append(metrics)

            status = "✅" if metrics["hit@k"] > 0 else "❌"
            diff_tag = f"[{difficulty}]" if difficulty else ""
            print(f"{status} Query {i}/{len(eval_queries)} {diff_tag}: MRR={metrics['mrr']:.2f}, Recall={metrics['recall@k']:.2f}")
        except Exception as e:
            print(f"❌ Query {i} failed: {e}")
            results.append({
                "query": query[:50],
                "strategy": strategy_name,
                "difficulty": difficulty,
                "query_type": query_type,
                "hit@k": 0, "recall@k": 0, "precision@k": 0,
                "mrr": 0, "map": 0, "ndcg@k": 0, "error": str(e)
            })

    df = pd.DataFrame(results)
    return df


def print_evaluation_summary(df: pd.DataFrame, strategy_name: str = "Strategy"):
    """Print aggregate metrics summary"""
    print(f"\n{'='*60}")
    print(f"📊 {strategy_name} - EVALUATION SUMMARY")
    print(f"{'='*60}")
    print(f"\nTotal Queries: {len(df)}")
    print(f"\n🎯 Retrieval Metrics (averaged):")
    print(f"  • Hit@k:       {df['hit@k'].mean():.3f}  (found at least 1 relevant)")
    print(f"  • Recall@k:    {df['recall@k'].mean():.3f}  (completeness)")
    print(f"  • Precision@k: {df['precision@k'].mean():.3f}  (accuracy)")
    print(f"  • MRR:         {df['mrr'].mean():.3f}  (first relevant rank)")
    print(f"  • MAP:         {df['map'].mean():.3f}  (ranking quality)")
    print(f"  • NDCG@k:      {df['ndcg@k'].mean():.3f}  (graded relevance)")
    print(f"\n✅ Success Rate: {df['hit@k'].mean()*100:.1f}%")
    print(f"{'='*60}")


print("✅ Evaluation utilities loaded")

✅ Evaluation utilities loaded


In [8]:
### Run Evaluation - Baseline Semantic Search

# Run evaluation for baseline (pure semantic search with InMemoryVectorStore)
df_baseline = evaluate_retrieval_strategy(
    baseline_semantic_retrieval,
    eval_queries,
    k=RETRIEVAL_K,
    strategy_name="Baseline (Semantic Only)"
)

# Print summary
print_evaluation_summary(df_baseline, "Baseline (Semantic Only)")


Evaluating: Baseline (Semantic Only) (k=5)
❌ Query 1/40 [easy]: MRR=0.00, Recall=0.00
✅ Query 2/40 [easy]: MRR=1.00, Recall=1.00
✅ Query 3/40 [easy]: MRR=1.00, Recall=1.00
✅ Query 4/40 [easy]: MRR=1.00, Recall=1.00
✅ Query 5/40 [easy]: MRR=0.50, Recall=1.00
✅ Query 6/40 [easy]: MRR=1.00, Recall=1.00
✅ Query 7/40 [easy]: MRR=1.00, Recall=1.00
✅ Query 8/40 [easy]: MRR=0.20, Recall=1.00
✅ Query 9/40 [easy]: MRR=1.00, Recall=1.00
✅ Query 10/40 [easy]: MRR=1.00, Recall=1.00
❌ Query 11/40 [easy]: MRR=0.00, Recall=0.00
✅ Query 12/40 [easy]: MRR=0.25, Recall=1.00
✅ Query 13/40 [easy]: MRR=0.50, Recall=1.00
✅ Query 14/40 [easy]: MRR=1.00, Recall=1.00
✅ Query 15/40 [easy]: MRR=1.00, Recall=1.00
✅ Query 16/40 [easy]: MRR=1.00, Recall=1.00
✅ Query 17/40 [easy]: MRR=1.00, Recall=1.00
✅ Query 18/40 [easy]: MRR=1.00, Recall=1.00
✅ Query 19/40 [easy]: MRR=0.50, Recall=1.00
✅ Query 20/40 [easy]: MRR=1.00, Recall=1.00
✅ Query 21/40 [medium]: MRR=0.20, Recall=0.17
✅ Query 22/40 [medium]: MRR=1.00, Recal

#### Results Summary Table

In [9]:
### Results Summary Table

# Display per-query results
print("\n📋 Per-Query Results:")
display(df_baseline[[
    "query", "difficulty", "hit@k", "recall@k", "precision@k", "mrr"
]])

# Aggregate metrics
comparison = df_baseline.groupby("strategy").agg({
    "hit@k": "mean",
    "recall@k": "mean",
    "precision@k": "mean",
    "mrr": "mean",
    "map": "mean",
    "ndcg@k": "mean",
}).round(3)

print("\n" + "=" * 80)
print("📊 BASELINE RETRIEVAL METRICS")
print("=" * 80)
display(comparison)


📋 Per-Query Results:


,query,difficulty,hit@k,recall@k,precision@k,mrr
0,Show me women's black sports shoes,easy,0.0,0.000000,0.0,0.000000
1,Is the Nike Women Free Run pink sports shoe av...,easy,1.0,1.000000,0.2,1.000000
2,Do you have ADIDAS women Ranger navy blue spor...,easy,1.0,1.000000,0.2,1.000000
3,Any women's red flats from Clarks?,easy,1.0,1.000000,0.2,1.000000
4,Show me iPanema white flip flops for women,easy,1.0,1.000000,0.2,0.500000
5,Women's purple casual shoes by Puma,easy,1.0,1.000000,0.2,1.000000
6,Carlton London nude formal heels for women,easy,1.0,1.000000,0.2,1.000000
7,Show me men's black flip flops from Fila,easy,1.0,1.000000,0.2,0.200000
8,Any ADIDAS black sandals for men?,easy,1.0,1.000000,0.2,1.000000
9,Is Buckaroo Men Flores black formal shoes avai...,easy,1.0,1.000000,0.2,1.000000



📊 BASELINE RETRIEVAL METRICS


,hit@k,recall@k,precision@k,mrr,map,ndcg@k
strategy,,,,,,
Baseline (Semantic Only),0.7,0.637,0.145,0.545,0.509,0.555


#### Detailed Results Per Query

In [10]:
### Detailed Results Per Query

def show_detailed_results(df: pd.DataFrame, strategy_name: str = "Strategy", max_show: int = 10):
    """Show detailed results for each query"""
    print(f"\n{'='*60}")
    print(f"📋 Detailed Results - {strategy_name}")
    print(f"{'='*60}")

    strategy_df = df[df["strategy"] == strategy_name].head(max_show)

    for i, row in strategy_df.iterrows():
        status = "✅" if row["hit@k"] > 0 else "❌"
        difficulty = row.get('difficulty', 'N/A')
        query_type = row.get('query_type', 'N/A')
        print(f"\n{status} [{difficulty}] Query: {row['query']}")
        print(f"   Type: {query_type}")
        print(f"   Expected: {row.get('ground_truth_ids', 'N/A')}")
        print(f"   Retrieved: {row.get('retrieved_ids', 'N/A')[:5] if row.get('retrieved_ids') else 'N/A'}")
        print(f"   Hit@k={row['hit@k']:.2f}, Recall={row['recall@k']:.2f}, MRR={row['mrr']:.2f}")

# Show detailed results
show_detailed_results(df_baseline, "Baseline (Semantic Only)", max_show=15)


📋 Detailed Results - Baseline (Semantic Only)

❌ [easy] Query: Show me women's black sports shoes
   Type: specific
   Expected: ['34651']
   Retrieved: ['42995', '36417', '9119', '9110', '42027']
   Hit@k=0.00, Recall=0.00, MRR=0.00

✅ [easy] Query: Is the Nike Women Free Run pink sports shoe availa...
   Type: availability
   Expected: ['36137']
   Retrieved: ['36137', '36136', '4149', '4148', '4579']
   Hit@k=1.00, Recall=1.00, MRR=1.00

✅ [easy] Query: Do you have ADIDAS women Ranger navy blue sports s...
   Type: specific
   Expected: ['34834']
   Retrieved: ['34834', '43366', '31971', '22163', '30259']
   Hit@k=1.00, Recall=1.00, MRR=1.00

✅ [easy] Query: Any women's red flats from Clarks?
   Type: specific
   Expected: ['10269']
   Retrieved: ['10269', '37968', '10309', '10307', '13704']
   Hit@k=1.00, Recall=1.00, MRR=1.00

✅ [easy] Query: Show me iPanema white flip flops for women
   Type: specific
   Expected: ['49809']
   Retrieved: ['44188', '49809', '41938', '44216', '419

#### Performance Analysis & Summary

In [11]:
### Performance by Difficulty Level

# Breakdown by difficulty level
if "difficulty" in df_baseline.columns:
    print("\n📊 Performance by Difficulty Level:")
    for difficulty in ["easy", "medium", "hard"]:
        diff_df = df_baseline[df_baseline["difficulty"] == difficulty]
        if len(diff_df) > 0:
            print(f"\n  [{difficulty.upper()}] ({len(diff_df)} queries)")
            print(f"    • Hit@k:    {diff_df['hit@k'].mean():.3f}")
            print(f"    • Recall:   {diff_df['recall@k'].mean():.3f}")
            print(f"    • MRR:      {diff_df['mrr'].mean():.3f}")

# Final summary
print("\n" + "=" * 80)
print("📊 BASELINE RETRIEVAL EVALUATION COMPLETE")
print("=" * 80)
print(f"\nConfiguration:")
print(f"  • Strategy: Baseline (Pure Semantic Search - InMemoryVectorStore)")
print(f"  • Embeddings: sentence-transformers/all-MiniLM-L6-v2")
print(f"  • k={RETRIEVAL_K}")
print(f"  • Total Queries: {len(df_baseline)}")
print(f"\n🎯 Overall Metrics:")
print(f"  • Hit@k:       {df_baseline['hit@k'].mean():.3f}")
print(f"  • Recall@k:    {df_baseline['recall@k'].mean():.3f}")
print(f"  • Precision@k: {df_baseline['precision@k'].mean():.3f}")
print(f"  • MRR:         {df_baseline['mrr'].mean():.3f}")
print(f"  • MAP:         {df_baseline['map'].mean():.3f}")
print(f"  • NDCG@k:      {df_baseline['ndcg@k'].mean():.3f}")
print(f"\n✅ Success Rate: {df_baseline['hit@k'].mean()*100:.1f}%")
print("=" * 80)


📊 Performance by Difficulty Level:

  [EASY] (24 queries)
    • Hit@k:    0.750
    • Recall:   0.750
    • MRR:      0.623

  [MEDIUM] (12 queries)
    • Hit@k:    0.750
    • Recall:   0.611
    • MRR:      0.489

  [HARD] (4 queries)
    • Hit@k:    0.250
    • Recall:   0.036
    • MRR:      0.250

📊 BASELINE RETRIEVAL EVALUATION COMPLETE

Configuration:
  • Strategy: Baseline (Pure Semantic Search - InMemoryVectorStore)
  • Embeddings: sentence-transformers/all-MiniLM-L6-v2
  • k=5
  • Total Queries: 40

🎯 Overall Metrics:
  • Hit@k:       0.700
  • Recall@k:    0.637
  • Precision@k: 0.145
  • MRR:         0.545
  • MAP:         0.509
  • NDCG@k:      0.555

✅ Success Rate: 70.0%
